In [ ]:
main_kaicheng.ipynb

In [ ]:
# ============================================================
# Compare five single neighbourhood-based features
# Each experiment adds only one additional feature
# to the baseline model for comparison
# ============================================================


def run_extra_feature(feature_name, f_train, f_val, f_test):
    # ------------------------------------------------------------
    # 1) Copy the original raw feature sets
    #    This prevents modification of the original data
    # ------------------------------------------------------------
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()
    Xte = X_test_raw.copy()

    # ------------------------------------------------------------
    # 2) Add the new neighbourhood-based feature
    # ------------------------------------------------------------
    Xtr[feature_name] = f_train
    Xva[feature_name] = f_val
    Xte[feature_name] = f_test

    # ------------------------------------------------------------
    # 3) Re-standardise the dataset after adding the new feature
    #    The scaler is fitted only on the training set
    #    to avoid data leakage
    # ------------------------------------------------------------
    sc = StandardScaler()

    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)
    Xte_s = sc.transform(Xte)

    # ------------------------------------------------------------
    # 4) Train a neural network model
    #    Use the same architecture as the baseline model
    #    to ensure a fair comparison
    # ------------------------------------------------------------
    

    baseline_model.fit(Xtr_s, y_train)

    # ------------------------------------------------------------
    # 5) Evaluate the model on validation and test sets
    # ------------------------------------------------------------
    baseline_val_pred = baseline_model.predict(Xva_s)
    baseline_test_pred = baseline_model.predict(Xte_s)

    val_mse = mean_squared_error(y_val, baseline_val_pred)
    test_mse = mean_squared_error(y_test, baseline_test_pred)
    return val_mse, test_mse


# ------------------------------------------------------------
# Run experiments for each neighbourhood feature separately
# ------------------------------------------------------------
single_feature_results = []

# 1) Mean price of K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMean",
    neigh_price_mean_train, neigh_price_mean_val, neigh_price_mean_test
)
single_feature_results.append(("NeighbourPriceMean", val_mse, test_mse))


# 2) Median price of K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMedian",
    neigh_price_median_train, neigh_price_median_val, neigh_price_median_test
)
single_feature_results.append(("NeighbourPriceMedian", val_mse, test_mse))


# 3) Minimum price among K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMin",
    neigh_price_min_train, neigh_price_min_val, neigh_price_min_test
)
single_feature_results.append(("NeighbourPriceMin", val_mse, test_mse))


# 4) Maximum price among K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMax",
    neigh_price_max_train, neigh_price_max_val, neigh_price_max_test
)
single_feature_results.append(("NeighbourPriceMax", val_mse, test_mse))


# 5) Price range among K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceRange",
    neigh_price_range_train, neigh_price_range_val, neigh_price_range_test
)
single_feature_results.append(("NeighbourPriceRange", val_mse, test_mse))


# ------------------------------------------------------------
# Convert results into a DataFrame for easy comparison
# and sort by validation MSE
# ------------------------------------------------------------
single_feature_results_df = pd.DataFrame(single_feature_results, columns=["Feature", "Val MSE", "Test MSE"])
single_feature_results_df = single_feature_results_df.sort_values("Val MSE").reset_index(drop=True)

print(single_feature_results_df)